In [2]:
import pandas as pd
import os 
import pickle as pkl

In [3]:
data_file= 'data/dataset.pkl'

In [4]:
#dictionary to map parties to num for tensors
party_dict={'SPÖ':0,'ÖVP':1,'FPÖ':2,'Grüne':3,'LIF':4,'BZÖ':5,'NEOS':6,'STRONACH':7,'PILZ':8,'independent':9}

In [5]:
df = pd.read_pickle(data_file)
df['speech_id'] = df.index
df['speech_id'] = df['speech_id'].astype(int)
independent_df= df[df['party']=='independent']
df = df[~(df['party']=='independent')]
#for testing I am using only part of data
#df= df[2000:3000]
#print(df['party'])
texts = df['text'].tolist()
df['party'] = df['party'].map(party_dict)
labels = df['party'].tolist()
test_texts= independent_df['text'].tolist()
mps = df['speaker'].tolist()
print(df.value_counts('party'))

party
0    24747
1    24063
2    17308
3    12455
5     4170
4     1942
6     1866
7     1322
8      346
dtype: int64


In [30]:
#df.to_excel('data/all_data.xlsx')

In [6]:
df.shape

(88219, 10)

In [7]:
mps = df['speaker'].unique()

In [8]:
print(mps)

['Heidrun Silhavy' 'Herbert Haupt' 'Walter Tancsits' 'Doris Bures'
 'Maximilian Walch' 'Eva Glawischnig-Piesczek' 'Ridi Steibl'
 'Rudolf Nürnberger' 'Sigisbert Dolinschek' 'Brigid Weinzinger'
 'Josef Cap' 'der Bellen' 'Wilhelm Molterer' 'Herbert Scheibner'
 'Wolfgang Schüssel' 'Alfred Gusenbauer' 'Friedrich Verzetnitsch'
 'Karl Öllinger' 'Magda Bleckmann' 'Reinhold Mitterlehner'
 'Michaela Sburny' 'Barbara Prammer' 'Karl Donabauer' 'Sabine Mandak'
 'Christoph Matznetter' 'Elisabeth Scheucher-Pichler' 'Uwe Scheuch'
 'Andrea Kuntzl' 'Reinhold Lopatka' 'Josef Broukal' 'Dieter Brosz'
 'Klaus Wittauer' 'Karin Hakl' 'Theresia Haidlmayr' 'Helene Partik-Pablé'
 'Karl Dobnigg' 'Karlheinz Kopf' 'Werner Kogler' 'Peter Pilz'
 'Josef Trinkl' 'Gabriela Moser' 'Jakob Auer' 'Günter Stummvoll'
 'Michael Spindelegger' 'Renate Csörgits' 'Elke Achleitner' 'Walter Posch'
 'Christine Marek' 'Marianne Hagenhofer' 'Detlev Neudeck' 'Franz Riepl'
 'Gabriele Heinisch-Hosek' 'Elmar Lichtenegger' 'Melitta Trunk'
 

In [9]:
speeches_by_party_dfs= [v for k,v in df.groupby('party')]

In [10]:
def sort_by_mp_frequency(df):
    df['n_speeches']=df.groupby('speaker')['speaker'].transform('count')
    df.sort_values(['n_speeches','speaker'],inplace=True,ascending=[False,True])
    return df

In [11]:
sorted_speeches_by_party_dfs = [sort_by_mp_frequency(df) for df in speeches_by_party_dfs]
num_k_folds=10

In [12]:
def distribute_mps(party_dfs,num_k_folds): 
    k_fold_dfs = {i: [] for i in range(num_k_folds)}
    for df in party_dfs:
        grouped_mps= df.groupby(['speaker'])
        for i, (mp, group) in enumerate(grouped_mps):
            target_df = i % num_k_folds
            k_fold_dfs[target_df].append(group)
    return [pd.concat(k_fold_dfs[i], ignore_index=True)for i in range(num_k_folds)]

k_fold_dfs = distribute_mps(sorted_speeches_by_party_dfs,num_k_folds)

In [13]:
#k_fold_file_name= "data/k_fold_dfs.pkl"
k_fold_file_name= "data/k_fold_dfs.pkl"

In [14]:
with open(k_fold_file_name, "wb") as file: 
    pkl.dump(k_fold_dfs,file)

In [15]:
with open(k_fold_file_name, "rb") as file:
    k_fold_dfs = pkl.load(file)

In [16]:
for i,df in enumerate(k_fold_dfs):
    df['k_fold']=i+1

In [17]:
df = pd.concat(k_fold_dfs)

In [18]:
df.to_excel("data/k_fold_data.xlsx")